# 02 · Ingesta de vuelos de EE. UU. (BTS On-Time Performance)

**Fuente:** Bureau of Transportation Statistics (Departamento de Transporte de EE. UU.), *Reporting Carrier On-Time
Performance*. Un archivo ZIP por mes con cada vuelo doméstico de las aerolíneas de EE. UU.: aeropuertos de origen y
destino (código IATA), salida y llegada **programadas y reales** (hora local de cada aeropuerto), demoras y cancelaciones.

**Para qué:** amplía el volumen real de entrenamiento del modelo con millones de vuelos de otro mercado.

**Qué hace este notebook:** descarga cada mes, comprueba el ZIP, extrae el CSV que trae dentro y lo guarda en bronze
**sin modificar su contenido**, con su manifiesto.

**Salida:** `bronze/bts/year=AAAA/month=MM/bts_AAAA_MM.csv` y `bronze/_manifiestos/bts/`.

In [ ]:
import os
from pathlib import Path

import yaml

# Carpeta del repositorio (este notebook está en notebooks/)
RAIZ = Path.cwd().parent
with open(RAIZ / "config" / "pipeline.yaml", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)
tablas = cfg["lake"]["tablas"]

# En la EC2 la variable LAKE_BUCKET trae el nombre del bucket; en una laptop se trabaja sobre data/lake
BUCKET = os.environ.get("LAKE_BUCKET", "").strip()
RAIZ_LOCAL = RAIZ / cfg["lake"]["raiz_local"]
BASE = f"s3a://{BUCKET}" if BUCKET else RAIZ_LOCAL.resolve().as_uri()


def ruta(tabla: str) -> str:
    """Ruta completa de una tabla declarada en config/pipeline.yaml."""
    return f"{BASE}/{tablas[tabla]}"


def meses(desde: str, hasta: str) -> list[tuple[int, int]]:
    """Pares (año, mes) entre dos fechas AAAA-MM, ambas incluidas."""
    anio, mes = map(int, desde.split("-"))
    anio_fin, mes_fin = map(int, hasta.split("-"))
    salida = []
    while (anio, mes) <= (anio_fin, mes_fin):
        salida.append((anio, mes))
        anio, mes = (anio + 1, 1) if mes == 12 else (anio, mes + 1)
    return salida


DESDE, HASTA = cfg["periodo"]["inicio"], cfg["periodo"]["fin"]
print("Destino:", f"s3://{BUCKET}" if BUCKET else "data/lake (local)")
print("Periodo:", DESDE, "a", HASTA)

In [ ]:
import zipfile

import pandas as pd

fuente = cfg["fuentes"]["bts"]

## Funciones de apoyo

Descargar con reintentos, calcular la huella (hash SHA-256) de cada archivo, subirlo a bronze y dejar su manifiesto.

In [ ]:
import hashlib
import json
import shutil
import tempfile
import time
from datetime import UTC, datetime

import boto3
import requests

s3 = boto3.client("s3") if BUCKET else None


def descargar(url: str, destino: Path, params: dict | None = None, reintentos: int = 5) -> bool:
    """Descarga por partes con reintentos de espera creciente. Devuelve False si el recurso no existe (404)."""
    cabeceras = {"User-Agent": cfg["fuentes"]["user_agent"]}
    for intento in range(1, reintentos + 1):
        try:
            with requests.get(url, params=params, headers=cabeceras, stream=True, timeout=300) as r:
                if r.status_code == 404:
                    return False
                r.raise_for_status()
                with open(destino, "wb") as f:
                    for bloque in r.iter_content(1 << 20):
                        f.write(bloque)
            return True
        except (requests.RequestException, OSError) as error:
            espera = min(5 * 2**intento, 300)
            print(f"  falló la descarga ({intento}/{reintentos}): {error}; reintento en {espera} s")
            time.sleep(espera)
    raise RuntimeError(f"No se pudo descargar {url}")


def sha256(ruta_archivo: Path) -> str:
    h = hashlib.sha256()
    with open(ruta_archivo, "rb") as f:
        for bloque in iter(lambda: f.read(1 << 20), b""):
            h.update(bloque)
    return h.hexdigest()


def contar_lineas(ruta_archivo: Path) -> int:
    with open(ruta_archivo, "rb") as f:
        return sum(bloque.count(b"\n") for bloque in iter(lambda: f.read(1 << 20), b""))


def existe(clave: str) -> bool:
    """True si la clave ya está en bronze."""
    if BUCKET:
        return s3.list_objects_v2(Bucket=BUCKET, Prefix=clave, MaxKeys=1).get("KeyCount", 0) > 0
    return (RAIZ_LOCAL / clave).exists()


def guardar(archivo: Path, clave: str) -> None:
    """Sube el archivo al bucket (o lo copia a data/lake en local)."""
    if BUCKET:
        s3.upload_file(str(archivo), BUCKET, clave)
    else:
        destino = RAIZ_LOCAL / clave
        destino.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(archivo, destino)


def escribir_manifiesto(fuente: str, clave: str, archivo: Path, url: str) -> dict:
    """Guarda de dónde salió el archivo, su tamaño, sus líneas, su hash y la fecha de descarga."""
    manifiesto = {
        "fuente": fuente, "archivo": clave, "url": url,
        "bytes": archivo.stat().st_size, "lineas": contar_lineas(archivo), "sha256": sha256(archivo),
        "descargado_utc": datetime.now(UTC).isoformat(timespec="seconds"),
    }
    temporal = archivo.with_name(archivo.name + ".json")
    temporal.write_text(json.dumps(manifiesto, ensure_ascii=False, indent=2), encoding="utf-8")
    guardar(temporal, f"{tablas['manifiestos']}/{fuente}/{archivo.name}.json")
    temporal.unlink()
    return manifiesto


def leer_manifiestos(fuente: str) -> list[dict]:
    """Todos los manifiestos de una fuente."""
    prefijo = f"{tablas['manifiestos']}/{fuente}/"
    if BUCKET:
        salida = []
        for pagina in s3.get_paginator("list_objects_v2").paginate(Bucket=BUCKET, Prefix=prefijo):
            for objeto in pagina.get("Contents", []):
                salida.append(json.loads(s3.get_object(Bucket=BUCKET, Key=objeto["Key"])["Body"].read()))
        return salida
    return [json.loads(p.read_text(encoding="utf-8")) for p in sorted((RAIZ_LOCAL / prefijo).glob("*.json"))]

## 1. URL de cada mes

In [ ]:
def url_mes(anio: int, mes: int) -> str:
    return fuente["url"].format(anio=anio, mes=mes)


url_mes(2024, 1)

## 2. Extracción y validación

El ZIP trae el CSV de datos y un `readme.html`. Se extrae solo el CSV y se comprueba que su encabezado tenga las
columnas de BTS que usará el pipeline.

In [ ]:
COLUMNAS_ESPERADAS = ["FlightDate", "Reporting_Airline", "Origin", "Dest", "CRSDepTime", "DepTime", "DepDelay", "Cancelled"]


def extraer_csv(zip_ruta: Path, destino: Path) -> None:
    if not zipfile.is_zipfile(zip_ruta):
        raise ValueError(f"{zip_ruta.name} no es un ZIP válido")
    with zipfile.ZipFile(zip_ruta) as z:
        csvs = [n for n in z.namelist() if n.lower().endswith(".csv")]
        if len(csvs) != 1:
            raise ValueError(f"{zip_ruta.name}: se esperaba un CSV y hay {csvs}")
        with z.open(csvs[0]) as origen, open(destino, "wb") as salida:
            while bloque := origen.read(1 << 20):
                salida.write(bloque)


def validar(ruta: Path) -> None:
    with open(ruta, encoding="utf-8", errors="replace") as f:
        encabezado = f.readline().replace('"', "").strip().split(",")
    faltan = [c for c in COLUMNAS_ESPERADAS if c not in encabezado]
    if faltan:
        raise ValueError(f"{ruta.name}: faltan columnas {faltan}")

## 3. Ingesta de un mes

Cada mes pesa unos 30 MB comprimido y unos 250 MB ya extraído. Se procesa de a uno para no llenar el disco.

In [ ]:
def ingerir_mes(anio: int, mes: int, forzar: bool = False) -> dict | None:
    clave = f"{tablas['bronze_bts']}/year={anio}/month={mes:02d}/bts_{anio}_{mes:02d}.csv"
    if existe(clave) and not forzar:
        print(f"{anio}-{mes:02d}: ya está en bronze")
        return None
    url = url_mes(anio, mes)
    with tempfile.TemporaryDirectory() as carpeta:
        zip_ruta = Path(carpeta) / "mes.zip"
        csv_ruta = Path(carpeta) / f"bts_{anio}_{mes:02d}.csv"
        if not descargar(url, zip_ruta):
            print(f"{anio}-{mes:02d}: BTS no lo tiene publicado")
            return None
        extraer_csv(zip_ruta, csv_ruta)
        validar(csv_ruta)
        guardar(csv_ruta, clave)
        manifiesto = escribir_manifiesto("bts", clave, csv_ruta, url)
    print(f"{anio}-{mes:02d}: {manifiesto['lineas']:,} líneas, {manifiesto['bytes'] / 1e6:.0f} MB")
    return manifiesto

## 4. Ejecutar el periodo completo

Son 36 meses y unos 9 GB en total: puede tardar entre 30 y 60 minutos. Si se interrumpe, al volver a ejecutar esta
celda continúa desde el primer mes que falta.

In [ ]:
for anio, mes in meses(DESDE, HASTA):
    ingerir_mes(anio, mes)

## 5. Verificación

In [ ]:
resumen = pd.DataFrame(leer_manifiestos("bts")).sort_values("archivo")
resumen["MB"] = (resumen["bytes"] / 1e6).round(0)
print(f"Archivos: {len(resumen)} · Líneas: {resumen['lineas'].sum():,} · Tamaño: {resumen['bytes'].sum() / 1e9:.2f} GB")
resumen[["archivo", "lineas", "MB", "descargado_utc"]]